In [ ]:
import os
from langsmith import traceable
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver # For Persistence
from langchain_openai import ChatOpenAI
from langgraph.types import interrupt, Command
from langchain_core.messages import (
    BaseMessage,
    SystemMessage, 
    HumanMessage
)

In [ ]:
# 1. State Definition
class State(TypedDict):
    # 'operator.add' allows appending messages rather than overwriting
    messages: Annotated[list[BaseMessage], lambda x, y: x + y]
    risk_score: float
    requires_approval: bool
    approved: bool

In [ ]:
# 2. Structured Output
class AnalyzerOutput(BaseModel):
    risk: Literal["low", "medium", "high"] = Field(description="Risk category for trade")
    reason: str = Field(description="What is your reasoning for assigning this category?")
        
# LLMs
model = ChatOpenAI(base_url='http://localhost:8080/v1', api_key="")
structured_model = model.with_structured_output(AnalyzerOutput)

In [ ]:
def analyzer_node(state: State):
    # Logic to evaluate trade risk
    prompt = [
        SystemMessage(content="You are a trade risk evaluator. If the user tries to sell more than 100,000 units, flag it as high risk."),
        HumanMessage(content = state["messages"][-1].content)
    ]
    msg = structured_model.invoke(prompt)
    # Simulating a risk check: if trade > $1M, set high risk
    response = {
        "messages": [msg.reason if msg else None],
        "risk_score": 0.1 if msg and msg.risk == "low" else (0.5 if msg and msg.risk == "medium" else 0.9), 
        "requires_approval": False,
        "approved": True
    }
    if response['risk_score'] > 0.5:
        response['requires_approval'] = True 
    return response

In [ ]:
def execution_node(state: State):
    if state['approved']:
        return {"messages": [HumanMessage(content="Trade executed successfully.")]}
    else:
        return {"messages": [HumanMessage(content="Trade rejected.")]}

In [ ]:
def human_gate(state: State):
    # 👇 Only pause if approval is required AND not yet approved
    state['approved'] = False
    if state["requires_approval"] == True and state.get('approved') is False:
        interrupt("High risk trade. Awaiting approval.")
    return state

In [ ]:
# 3. Build Graph with Persistence
workflow = StateGraph(State)
workflow.add_node("analyzer", analyzer_node)
workflow.add_node("executor", execution_node)
workflow.add_node("human_gate", human_gate) 
workflow.set_entry_point("analyzer")

In [ ]:
# Conditional Edge: Only proceed if risk is low
workflow.add_conditional_edges(
    "analyzer",
    lambda x: "executor" if x["risk_score"] < 0.5 else "human_gate"
)
workflow.add_edge("human_gate", "executor")

# 4. The "Secret Sauce": Checkpointer and Breakpoints
app = workflow.compile(checkpointer=MemorySaver())

In [ ]:

# --- RUNNING THE SYSTEM ---
config = {"configurable": {"thread_id": "123"}}
initial_input = {"messages": [HumanMessage(content="Sell 20,000 units of Asset X")]}

# Stage 1: The AI analyzes and hits the breakpoint
print("--- Starting AI Analysis ---")
# 1. Stream until interrupt
for event in app.stream(initial_input, config, stream_mode="values"):
    print(event)

# 2. Graph is now paused — check state and resume
if app.get_state(config).values["requires_approval"] == True:
    decision = input("Approve? (approved/rejected): ").strip().lower()
    app.invoke(Command(resume={"approved": decision == "approved"}), config)

print("Process Complete.")